# Data and Feature Foundations

🎯 **Purpose**: Clean the dataset and create a reusable feature table per contestant per episode

### Imports

In [27]:
from pathlib import Path
import pandas as pd

## Inspect the data

In [2]:
df = pd.read_csv("../data/raw/episode_scores.csv")
df.head(20)

,id,episode,episode_label,contestant,contestant_label,score,rank,series,srank
0,1,1,Melon buffet.,3,Frank Skinner,19,1=,19,1=
1,2,1,Melon buffet.,4,Josh Widdicombe,13,4,13,4
2,3,1,Melon buffet.,5,Roisin Conaty,7,5,7,5
3,4,1,Melon buffet.,6,Romesh Ranganathan,19,1=,19,1=
4,5,1,Melon buffet.,7,Tim Key,17,3,17,3
5,6,2,The pie whisperer.,3,Frank Skinner,9,5,28,4=
6,7,2,The pie whisperer.,4,Josh Widdicombe,16,3,29,3
7,8,2,The pie whisperer.,5,Roisin Conaty,21,1,28,4=
8,9,2,The pie whisperer.,6,Romesh Ranganathan,14,4,33,2
9,10,2,The pie whisperer.,7,Tim Key,18,2,35,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 730 entries, 0 to 729
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                730 non-null    int64 
 1   episode           730 non-null    int64 
 2   episode_label     730 non-null    object
 3   contestant        730 non-null    int64 
 4   contestant_label  730 non-null    object
 5   score             730 non-null    int64 
 6   rank              730 non-null    object
 7   series            730 non-null    int64 
 8   srank             730 non-null    object
dtypes: int64(5), object(4)
memory usage: 51.5+ KB


In [4]:
df.describe()

,id,episode,contestant,score,series
count,730.000000,730.000000,730.000000,730.000000,730.000000
mean,365.500000,73.500000,55.000000,15.490411,76.634247
std,210.877136,42.174478,29.173809,4.242711,45.277128
min,1.000000,1.000000,3.000000,3.000000,5.000000
25%,183.250000,37.000000,34.000000,13.000000,36.000000
50%,365.500000,73.500000,52.000000,16.000000,73.000000
75%,547.750000,110.000000,79.000000,18.000000,112.000000
max,730.000000,146.000000,107.000000,30.000000,184.000000


In [5]:
df.duplicated().sum()

np.int64(0)

In [6]:
df["series"].nunique()
df["series"].sort_values().unique()[:20]

array([ 5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21,
       22, 23, 24])

In [7]:
df.groupby(["series", "episode"]).size().value_counts()

1    635
2     44
4      1
3      1
Name: count, dtype: int64

## Structure the data

In [8]:
task_df = df.copy()

# Only keep a few columns and rename
task_df = task_df[[
    "episode",
    "episode_label",
    "contestant_label",
    "score"
]]

task_df = task_df.rename(columns={
    "contestant_label": "contestant",
    "score": "episode_score"
})

# Sort for time-series logic
task_df = task_df.sort_values(["episode", "contestant"])

task_df.head(10)

,episode,episode_label,contestant,episode_score
0,1,Melon buffet.,Frank Skinner,19
1,1,Melon buffet.,Josh Widdicombe,13
2,1,Melon buffet.,Roisin Conaty,7
3,1,Melon buffet.,Romesh Ranganathan,19
4,1,Melon buffet.,Tim Key,17
5,2,The pie whisperer.,Frank Skinner,9
6,2,The pie whisperer.,Josh Widdicombe,16
7,2,The pie whisperer.,Roisin Conaty,21
8,2,The pie whisperer.,Romesh Ranganathan,14
9,2,The pie whisperer.,Tim Key,18


### Creating series groups

In [ ]:
# Create contestant group per episode
episode_groups = (
    task_df.groupby("episode")["contestant"].apply(lambda x: tuple(sorted(x)))
)

# Turn groups into series ID numbers (each series = an exact group of contestants)
series_map = episode_groups.factorize()[0]

# Map new series IDs back to df
task_df["series_id"] = task_df["episode"].map(
    dict(zip(episode_groups.index, series_map))
)

# Adjust so series 1 starts at ID 1
task_df["series_id"] += 1

# Check how many contestants per series
task_df.groupby(["series_id", "episode"]).size().value_counts()

5    146
Name: count, dtype: int64

In [11]:
task_df.head(20)

,episode,episode_label,contestant,episode_score,series_id
0,1,Melon buffet.,Frank Skinner,19,1
1,1,Melon buffet.,Josh Widdicombe,13,1
2,1,Melon buffet.,Roisin Conaty,7,1
3,1,Melon buffet.,Romesh Ranganathan,19,1
4,1,Melon buffet.,Tim Key,17,1
5,2,The pie whisperer.,Frank Skinner,9,1
6,2,The pie whisperer.,Josh Widdicombe,16,1
7,2,The pie whisperer.,Roisin Conaty,21,1
8,2,The pie whisperer.,Romesh Ranganathan,14,1
9,2,The pie whisperer.,Tim Key,18,1


In [ ]:
# Check number of series
task_df["series_id"].nunique()

21

In [ ]:
# Check out the first series
task_df[task_df["series_id"] == 1].sort_values("episode")

,episode,episode_label,contestant,episode_score,series_id
0,1,Melon buffet.,Frank Skinner,19,1
1,1,Melon buffet.,Josh Widdicombe,13,1
2,1,Melon buffet.,Roisin Conaty,7,1
3,1,Melon buffet.,Romesh Ranganathan,19,1
4,1,Melon buffet.,Tim Key,17,1
5,2,The pie whisperer.,Frank Skinner,9,1
6,2,The pie whisperer.,Josh Widdicombe,16,1
7,2,The pie whisperer.,Roisin Conaty,21,1
8,2,The pie whisperer.,Romesh Ranganathan,14,1
9,2,The pie whisperer.,Tim Key,18,1


In [ ]:
# Add episode ID within a series
task_df["episode_in_series"] = task_df.groupby("series_id")["episode"] \
    .rank(method="dense").astype(int)

task_df.head(20)

,episode,episode_label,contestant,episode_score,series_id,episode_in_series
0,1,Melon buffet.,Frank Skinner,19,1,1
1,1,Melon buffet.,Josh Widdicombe,13,1,1
2,1,Melon buffet.,Roisin Conaty,7,1,1
3,1,Melon buffet.,Romesh Ranganathan,19,1,1
4,1,Melon buffet.,Tim Key,17,1,1
5,2,The pie whisperer.,Frank Skinner,9,1,2
6,2,The pie whisperer.,Josh Widdicombe,16,1,2
7,2,The pie whisperer.,Roisin Conaty,21,1,2
8,2,The pie whisperer.,Romesh Ranganathan,14,1,2
9,2,The pie whisperer.,Tim Key,18,1,2


## Create features

#### Running total

In [18]:
task_df["cumulative_score"] = task_df.groupby(["series_id", "contestant"])["episode_score"].cumsum()

#### Mean score so far

In [ ]:
task_df["mean_score_so_far"] = (
    task_df.groupby(["series_id", "contestant"])["episode_score"]
    .expanding().mean()
    .reset_index(level=[0,1], drop=True)
)

#### Standard Deviation score so far - aka *Chaos Factor*

In [ ]:
task_df["std_score_so_far"] = (
    task_df.groupby(["series_id", "contestant"])["episode_score"]
    .expanding().std()
    .reset_index(level=[0,1], drop=True)
)

# Replace NaN with 0 (first episode has no std yet)
task_df["std_score_so_far"] = task_df["std_score_so_far"].fillna(0)


#### Rolling average of last 3 episodes aka *Recent Form*

In [22]:
task_df["recent_avg_score"] = (
    task_df.groupby(["series_id", "contestant"])["episode_score"]
    .rolling(window=2).mean()
    .reset_index(level=[0,1], drop=True)
)

# For first episode, fallback to current score
task_df["recent_avg_score"] = task_df["recent_avg_score"].fillna(task_df["episode_score"])

#### Momentum - are contestants getting better or worse?

In [23]:
task_df["momentum"] = task_df["recent_avg_score"] - task_df["mean_score_so_far"]

#### For context: Episodes Played

In [24]:
task_df["episodes_played"] = task_df.groupby(["series_id", "contestant"]).cumcount() + 1

In [25]:
# Check outputs
task_df[
    (task_df["series_id"] == 4) &
    (task_df["contestant"] == "Mel Giedroyc")
].head(10)

,episode,episode_label,contestant,episode_score,series_id,episode_in_series,cumulative_score,mean_score_so_far,std_score_so_far,recent_avg_score,momentum,episodes_played
83,17,A fat bald white man.,Mel Giedroyc,16,4,1,16,16.000000,0.000000,16.0,0.000000,1
88,18,Look at me.,Mel Giedroyc,20,4,2,36,18.000000,2.828427,18.0,0.000000,2
93,19,Hollowing out a baguette.,Mel Giedroyc,3,4,3,39,13.000000,8.888194,11.5,-1.500000,3
98,20,Friendship is truth.,Mel Giedroyc,24,4,4,63,15.750000,9.105859,13.5,-2.250000,4
103,21,Meat.,Mel Giedroyc,21,4,5,84,16.800000,8.228001,22.5,5.700000,5
108,22,Spatchcock it.,Mel Giedroyc,16,4,6,100,16.666667,7.366591,18.5,1.833333,6
113,23,No stars for naughty boys.,Mel Giedroyc,13,4,7,113,16.142857,6.866066,14.5,-1.642857,7
118,24,Tony Three Pies.,Mel Giedroyc,21,4,8,134,16.750000,6.584614,17.0,0.250000,8


👉 Raw scores have now been converted into a dynamic representation of contestant performance over time, including new **momentum**, **chaos factor** and **recent form** features.

## Save clean dataset

In [28]:
task_clean_df = task_df.copy()

output_dir = Path("../data/processed")
task_clean_df.to_csv(output_dir / "task_clean_df.csv", index=False)

# ---------- Confirm file saved ----------
print("Saved file:")
print(output_dir / "task_clean_df.csv")
print()
print(f"task_clean_df shape: {task_clean_df.shape}")

Saved file:
../data/processed/task_clean_df.csv

task_clean_df shape: (730, 12)
